In [5]:
import os
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
from concurrent.futures import ThreadPoolExecutor
import math


PARQUET_PATH = "data/pubchem_100K.parquet"  
SMILES_COL   = "SMILES"          

df = pd.read_parquet(PARQUET_PATH)
print(df.shape)
df.head()


(100000, 1)


,SMILES
0,CN(c1ccccc1)c1ccccc1C(=O)NCC1(O)CCOCC1
1,CC[NH+](CC)C1CCC([NH2+]C2CC2)(C(=O)[O-])C1
2,COCC(CNC(=O)c1ccc2c(c1)NC(=O)C2)OC
3,OCCn1cc(CNc2cccc3c2CCCC3)nn1
4,CCCCCCc1ccc(C#Cc2ccc(C#CC3=CC=C(CCC)CC3)c(C3CC...


In [6]:
def canonicalize_smiles(smiles: str):
    """
    Takes a SMILES string, returns RDKit canonical SMILES.
    Returns None for invalid or missing inputs.
    """
    if smiles is None or (isinstance(smiles, float) and math.isnan(smiles)):
        return None
    
    if not isinstance(smiles, str) or smiles.strip() == "":
        return None
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # RDKit's canonical SMILES
    return Chem.MolToSmiles(mol, canonical=True)


In [7]:
smiles_list = df[SMILES_COL].tolist()

num_workers = os.cpu_count() or 4  # sensible default if detection fails
print(f"Using {num_workers} threads")

# ThreadPoolExecutor is fine here because RDKit's heavy work is in C++ and releases the GIL
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    canonical_list = list(executor.map(canonicalize_smiles, smiles_list, chunksize=500))

# Attach back to the DataFrame
canonical_df = pd.DataFrame({
    SMILES_COL: canonical_list
})

# Optionally drop invalid ones (where canonicalization failed)
canonical_df = canonical_df.dropna(subset=[SMILES_COL]).reset_index(drop=True)


Using 8 threads


In [8]:
canonical_df.head()

,SMILES
0,CN(c1ccccc1)c1ccccc1C(=O)NCC1(O)CCOCC1
1,CC[NH+](CC)C1CCC([NH2+]C2CC2)(C(=O)[O-])C1
2,COCC(CNC(=O)c1ccc2c(c1)NC(=O)C2)OC
3,OCCn1cc(CNc2cccc3c2CCCC3)nn1
4,CCCCCCc1ccc(C#Cc2ccc(C#CC3=CC=C(CCC)CC3)c(C3CC...


In [11]:
canonical_df.to_parquet("data/pubchem_100k_canonical.parquet", index=False)
print(f"Saved canonicalized SMILES")


Saved canonicalized SMILES
